# services.management

> Service layer wrapping `cjm-workflow-state` for session management operations

In [ ]:
#| default_exp services.management

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, List, Optional

from fastcore.basics import patch

from cjm_workflow_state.state_store import SQLiteWorkflowStateStore, SessionSummary

from cjm_fasthtml_workflow_session_management.models import (
    EnrichedSessionSummary,
    SessionEnricher,
    LabelGenerator,
)
from cjm_fasthtml_workflow_session_management.utils import default_label

## Debug Flag

In [ ]:
#| export
DEBUG_SESSION_SERVICE = False # Enable for verbose session service logging

## SessionManagementService

Core class with constructor, private helpers for label resolution and enrichment, and public methods added via `@patch` in subsequent cells. The service is scoped to a single `flow_id` — hosts instantiate one service per workflow they want to manage.

In [ ]:
#| export
class SessionManagementService:
    """Service wrapping cjm-workflow-state for session listing and lifecycle operations."""
    
    def __init__(
        self,
        state_store:SQLiteWorkflowStateStore, # The workflow state store to manage sessions in
        flow_id:str, # Workflow identifier this service is scoped to
        enricher:Optional[SessionEnricher]=None, # Optional host callback that enriches list display
        label_generator:Optional[LabelGenerator]=None, # Optional host callback for default labels
    ):
        """Initialize with a state store, flow ID, and optional host callbacks."""
        self._store = state_store
        self._flow_id = flow_id
        self._enricher = enricher
        self._label_generator = label_generator
    
    @property
    def flow_id(self) -> str: # The flow ID this service manages
        """The workflow identifier this service is scoped to."""
        return self._flow_id
    
    def _resolve_label(
        self,
        summary:SessionSummary, # Raw session metadata
        state_json:Dict[str, Any] # Full state blob for label generator input
    ) -> str: # Resolved display label
        """Resolve the display label: stored label > host generator > default."""
        if summary.label:
            return summary.label
        if self._label_generator is not None:
            try:
                return self._label_generator(summary, state_json)
            except Exception as e:
                if DEBUG_SESSION_SERVICE:
                    print(f"[SessionManagementService] label_generator raised: {e}")
        return default_label(summary.created_at)
    
    def _enrich(
        self,
        state_json:Dict[str, Any] # Full state blob for enricher input
    ) -> Dict[str, str]: # Enricher fields, empty dict if no enricher or on error
        """Run the host enricher against a state blob, returning an empty dict on any failure."""
        if self._enricher is None:
            return {}
        try:
            result = self._enricher(state_json)
            # Coerce all values to strings defensively.
            return {k: str(v) for k, v in result.items()}
        except Exception as e:
            if DEBUG_SESSION_SERVICE:
                print(f"[SessionManagementService] enricher raised: {e}")
            return {}
    
    def _to_enriched(
        self,
        summary:SessionSummary # Raw session metadata
    ) -> EnrichedSessionSummary: # Summary + label + enricher fields
        """Load the session's state blob and combine it with label resolution + enrichment."""
        state_json = self._store.get_state(self._flow_id, summary.session_id)
        return EnrichedSessionSummary(
            summary=summary,
            resolved_label=self._resolve_label(summary, state_json),
            enriched_fields=self._enrich(state_json),
        )

### List Sessions

The core read path for the manager UI. Returns one `EnrichedSessionSummary` per session in the managed flow, with labels resolved and host enricher applied.

Note: the service is **sync**, not async. The state store is a local SQLite DB — async wrapping would be overhead without benefit. The reference library's async methods exist because *its* operations cross plugin process boundaries.

In [ ]:
#| export
@patch
def list_sessions(
    self:SessionManagementService,
    order_by:str="updated_at", # Sort column: "updated_at", "created_at", or "label"
    descending:bool=True # Sort direction
) -> List[EnrichedSessionSummary]: # All sessions with labels resolved and enricher applied
    """List all sessions for the managed flow with display enrichment applied."""
    summaries = self._store.list_sessions(self._flow_id, order_by=order_by, descending=descending)
    if DEBUG_SESSION_SERVICE:
        print(f"[SessionManagementService] list_sessions: {len(summaries)} sessions for flow={self._flow_id}")
    return [self._to_enriched(s) for s in summaries]

### Get Session

Single-session read path for detail views or to verify a session exists before acting on it.

In [ ]:
#| export
@patch
def get_session(
    self:SessionManagementService,
    session_id:str # Session identifier
) -> Optional[EnrichedSessionSummary]: # Enriched session, or None if not found
    """Fetch a single enriched session summary by ID."""
    summary = self._store.get_session_summary(self._flow_id, session_id)
    if summary is None:
        return None
    return self._to_enriched(summary)

@patch
def session_exists(
    self:SessionManagementService,
    session_id:str # Session identifier
) -> bool: # True if a row exists
    """Check whether a session exists in the managed flow."""
    return self._store.session_exists(self._flow_id, session_id)

### Create Session

Mints a new session row in the managed flow. The caller is expected to switch the HTTP session's active workflow-session-ID to this new ID via `cjm-fasthtml-interactions.set_session_id()` if they want the workflow to start using it.

In [ ]:
#| export
@patch
def create_session(
    self:SessionManagementService,
    label:Optional[str]=None # Optional human-readable label for the new session
) -> str: # The new session_id (auto-generated UUID4)
    """Create a new empty session and return its session_id."""
    session_id = self._store.create_session(self._flow_id, label=label)
    if DEBUG_SESSION_SERVICE:
        print(f"[SessionManagementService] created session {session_id[:8]}... in flow={self._flow_id}")
    return session_id

### Rename and Delete

Both are thin wrappers over the state store. Rename is a no-op on missing sessions (matching the state store's semantics); delete is idempotent.

In [ ]:
#| export
@patch
def rename_session(
    self:SessionManagementService,
    session_id:str, # Session identifier
    label:Optional[str] # New label, or None to clear
) -> None:
    """Update a session's label. No-op if the session does not exist."""
    self._store.set_session_label(self._flow_id, session_id, label)
    if DEBUG_SESSION_SERVICE:
        print(f"[SessionManagementService] renamed {session_id[:8]}... to {label!r}")

@patch
def delete_session(
    self:SessionManagementService,
    session_id:str # Session identifier
) -> None:
    """Delete a session row. Idempotent."""
    self._store.delete_session(self._flow_id, session_id)
    if DEBUG_SESSION_SERVICE:
        print(f"[SessionManagementService] deleted {session_id[:8]}...")

## Tests

Two fixtures: an isolated temp DB for CRUD tests, and the real `test_files/workflow_state.db` copy from the decomp workflow for enrichment validation against production-shaped state.

In [ ]:
import tempfile, os

_tmp = tempfile.NamedTemporaryFile(suffix='.db', delete=False)
_tmp_path = _tmp.name
_tmp.close()
temp_store = SQLiteWorkflowStateStore(_tmp_path)

# Seed three sessions with differing labels and state.
sid1 = temp_store.create_session("decomp", label="First")
temp_store.update_state("decomp", sid1, {
    "step_states": {"selection": {"selected_sources": [{"media_path": "/audio/a.wav"}]}}
})
import time; time.sleep(0.01)

sid2 = temp_store.create_session("decomp")  # no label — should fall through to generator
temp_store.update_state("decomp", sid2, {
    "step_states": {"selection": {"selected_sources": [
        {"media_path": "/audio/b.wav"}, {"media_path": "/audio/c.wav"}
    ]}}
})
time.sleep(0.01)

sid3 = temp_store.create_session("decomp")  # no label, no state — should fall through to default
print(f"Seeded: {sid1[:8]}, {sid2[:8]}, {sid3[:8]}")

Seeded: 97151f31, 8bb41498, 495b8f58


In [ ]:
# Service with no enricher / label_generator: stored labels and defaults.
plain_service = SessionManagementService(temp_store, "decomp")
sessions = plain_service.list_sessions()
assert len(sessions) == 3

# sid1 has stored label "First".
by_id = {s.summary.session_id: s for s in sessions}
assert by_id[sid1].resolved_label == "First"
assert by_id[sid1].enriched_fields == {}  # no enricher

# sid2 and sid3 fall through to the default label.
assert by_id[sid2].resolved_label.startswith("Session ")
assert by_id[sid3].resolved_label.startswith("Session ")

# Ordering: updated_at DESC, so sid3 (newest) should be first.
assert sessions[0].summary.session_id == sid3
assert sessions[-1].summary.session_id == sid1

print(f"Plain service OK: {[s.resolved_label for s in sessions]}")

Plain service OK: ['Session 2026-04-08 22:34', 'Session 2026-04-08 22:34', 'First']


In [ ]:
# Service with enricher and label_generator.
def _test_enricher(state_json):
    sources = state_json.get("step_states", {}).get("selection", {}).get("selected_sources", [])
    return {"sources": str(len(sources))}

def _test_label_gen(summary, state_json):
    sources = state_json.get("step_states", {}).get("selection", {}).get("selected_sources", [])
    if sources:
        from pathlib import Path
        return Path(sources[0]["media_path"]).stem
    return "Empty session"

enriched_service = SessionManagementService(
    temp_store, "decomp",
    enricher=_test_enricher,
    label_generator=_test_label_gen,
)
sessions = enriched_service.list_sessions()
by_id = {s.summary.session_id: s for s in sessions}

# Enricher applied to all: sid1 has 1 source, sid2 has 2, sid3 has 0.
assert by_id[sid1].enriched_fields == {"sources": "1"}
assert by_id[sid2].enriched_fields == {"sources": "2"}
assert by_id[sid3].enriched_fields == {"sources": "0"}

# Label resolution:
# - sid1 has stored label "First" → stored wins over generator.
# - sid2 has no label → generator produces "b" from /audio/b.wav stem.
# - sid3 has no label, no sources → generator returns "Empty session".
assert by_id[sid1].resolved_label == "First"
assert by_id[sid2].resolved_label == "b"
assert by_id[sid3].resolved_label == "Empty session"

print(f"Enricher + label_gen OK: {[(s.resolved_label, s.enriched_fields) for s in sessions]}")

Enricher + label_gen OK: [('Empty session', {'sources': '0'}), ('b', {'sources': '2'}), ('First', {'sources': '1'})]


In [ ]:
# Misbehaving callbacks must not take down the list.
def _broken_enricher(state_json):
    raise RuntimeError("boom")

def _broken_label_gen(summary, state_json):
    raise ValueError("kaboom")

broken_service = SessionManagementService(
    temp_store, "decomp",
    enricher=_broken_enricher,
    label_generator=_broken_label_gen,
)
sessions = broken_service.list_sessions()
assert len(sessions) == 3
# Enricher failure → empty dict, not a crash.
assert all(s.enriched_fields == {} for s in sessions)
# sid1 still uses stored label (generator was never called).
by_id = {s.summary.session_id: s for s in sessions}
assert by_id[sid1].resolved_label == "First"
# sid2 / sid3 would call the broken generator → fall through to default.
assert by_id[sid2].resolved_label.startswith("Session ")
assert by_id[sid3].resolved_label.startswith("Session ")

print("Broken-callback resilience OK")

Broken-callback resilience OK


In [ ]:
# get_session + session_exists.
one = plain_service.get_session(sid1)
assert one is not None
assert one.summary.session_id == sid1
assert one.resolved_label == "First"
assert plain_service.session_exists(sid1)

missing = plain_service.get_session("does-not-exist")
assert missing is None
assert not plain_service.session_exists("does-not-exist")

print("get_session / session_exists OK")

get_session / session_exists OK


In [ ]:
# create_session / rename_session / delete_session lifecycle.
new_id = plain_service.create_session(label="Freshly Minted")
assert plain_service.session_exists(new_id)
assert plain_service.get_session(new_id).resolved_label == "Freshly Minted"

plain_service.rename_session(new_id, "Renamed")
assert plain_service.get_session(new_id).resolved_label == "Renamed"

plain_service.rename_session(new_id, None)  # clear label — falls back to default
assert plain_service.get_session(new_id).resolved_label.startswith("Session ")

plain_service.delete_session(new_id)
assert not plain_service.session_exists(new_id)

# Idempotent delete.
plain_service.delete_session(new_id)
plain_service.delete_session("never existed")

print("CRUD lifecycle OK")

CRUD lifecycle OK


In [ ]:
# flow_id isolation: sessions in other flows are invisible.
other_service = SessionManagementService(temp_store, "other_flow")
other_sessions = other_service.list_sessions()
assert other_sessions == []

# Flow isolation on CRUD: creating a session in "other_flow" must not appear in "decomp".
other_id = other_service.create_session(label="Outsider")
assert len(plain_service.list_sessions()) == 3  # unchanged
assert len(other_service.list_sessions()) == 1

# Cleanup.
other_service.delete_session(other_id)
os.unlink(_tmp_path)
print("Flow isolation OK")

Flow isolation OK


### Production Fixture Test

Validates the service against a real `workflow_state.db` copied from the decomposition workflow host. Proves label resolution and enrichment work on real state shapes (not just synthetic fixtures), and exercises the label-column migration on a realistic DB.

In [ ]:
import shutil
from pathlib import Path

# Work on a copy so the fixture stays pristine for future test runs.
# Note: the fixture file itself may be mutated by the demo app between test runs
# (creating, renaming, deleting sessions). The assertions below therefore only
# validate that the service *works* against real-shaped data, not that the
# fixture is in any particular configuration.
_fixture_src = Path("../../test_files/workflow_state.db")
if _fixture_src.exists():
    _fix_copy = tempfile.NamedTemporaryFile(suffix='.db', delete=False)
    _fix_copy_path = _fix_copy.name
    _fix_copy.close()
    shutil.copy2(_fixture_src, _fix_copy_path)
    
    prod_store = SQLiteWorkflowStateStore(_fix_copy_path)
    
    # A realistic enricher matching the planned decomp enricher shape.
    def _decomp_enricher(state_json):
        step_states = state_json.get("step_states", {})
        sources = step_states.get("selection", {}).get("selected_sources", [])
        seg_state = step_states.get("segmentation", {}) or step_states.get("decomposition", {})
        segments = seg_state.get("segments", [])
        return {
            "sources": str(len(sources)) if sources else "—",
            "segments": str(len(segments)) if segments else "—",
        }
    
    def _decomp_label_gen(summary, state_json):
        sources = state_json.get("step_states", {}).get("selection", {}).get("selected_sources", [])
        if sources:
            first = sources[0].get("media_path") or sources[0].get("label", "")
            if first:
                return Path(first).stem
        return default_label(summary.created_at)
    
    prod_service = SessionManagementService(
        prod_store, "structure_decomposition",
        enricher=_decomp_enricher,
        label_generator=_decomp_label_gen,
    )
    
    prod_sessions = prod_service.list_sessions()
    print(f"Fixture has {len(prod_sessions)} sessions in structure_decomposition")
    for s in prod_sessions:
        print(f"  {s.summary.session_id[:8]}... step={s.summary.current_step!r} "
              f"label={s.resolved_label!r} enriched={s.enriched_fields} "
              f"size={s.summary.state_size_bytes}B")
    
    # Structural assertions — validate the service works, without assuming a pristine fixture state.
    assert isinstance(prod_sessions, list)
    # If there are any sessions, each one should have both an enricher fields dict
    # and a resolved label string (never None, never empty).
    for s in prod_sessions:
        assert isinstance(s.enriched_fields, dict)
        assert "sources" in s.enriched_fields
        assert "segments" in s.enriched_fields
        assert isinstance(s.resolved_label, str) and s.resolved_label, \
            f"empty resolved_label on {s.summary.session_id[:8]}..."
        # Size hint should be non-negative.
        assert s.summary.state_size_bytes >= 0
    
    os.unlink(_fix_copy_path)
    print("Production fixture tests passed!")
else:
    print(f"(Skipping fixture test — {_fixture_src} not found)")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()